## Assess the Kronos Tokenizer

This notebook will be used to assess the quality of the Kronos tokens. The encoder and decoder will be run **once** (using RUN_ENCODER and RUN_DECODER flag = True). The results will be saved to the drive  *"/Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/final_model/tokens"* here and then on any subsequent run we set the flags to false and just load the existing files. The first 60 time points of each trading day for each asset/channel will **not** be tokenized since we need a context of at least 60. Every other time point duing the day will be tokenized. 

# Prelims 

In [ ]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.models.kronos_tokenizer import (
    KronosTokenizerAdapter,
    decode_causal_split,
    encode_causal_split,
)
from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.utils.config import load_yaml
from src.evaluation.tokenizer_metrics import plot_tokenizer_reconstruction



Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [2]:
RUN_ENCODER = True
RUN_DECODER = True

CONTEXT_LENGTH = 60
WINDOW_BATCH_SIZE = 8
SERIES_BATCH_SIZE = 93

DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

TOKEN_CACHE_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/final_model/tokens"
)

ENCODED_DATA_PATH = (
    TOKEN_CACHE_DIR / "encoded_data.pt"
)

DECODED_DATA_PATH = (
    TOKEN_CACHE_DIR / "decoded_data.pt"
)

TOKEN_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def save_torch_atomic(
    value: object,
    path: Path,
) -> None:
    temporary_path = path.with_name(
        f"{path.name}.tmp"
    )

    torch.save(
        value,
        temporary_path,
    )

    temporary_path.replace(path)

# Load the data

In [3]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

# Load the tokenizer

In [4]:
kronos_tokenizer = (
    KronosTokenizerAdapter.from_config(
        config,
        series_batch_size=SERIES_BATCH_SIZE,
    )
    .load()
)

print("Loaded frozen Kronos tokenizer on CPU.")

Loaded frozen Kronos tokenizer on CPU.


In [ ]:
if RUN_ENCODER:
    encoded_data = encode_causal_split(
        kronos_tokenizer,
        train,
        context_length=CONTEXT_LENGTH,
        window_batch_size=WINDOW_BATCH_SIZE,
        series_batch_size=SERIES_BATCH_SIZE,
        show_progress=True,
    )

    save_torch_atomic(
        encoded_data,
        ENCODED_DATA_PATH,
    )

    print(
        "Saved encoded data to:",
        ENCODED_DATA_PATH,
    )

else:
    if not ENCODED_DATA_PATH.exists():
        raise FileNotFoundError(
            "RUN_ENCODER is False but the encoded "
            f"cache does not exist: {ENCODED_DATA_PATH}"
        )

    encoded_data = torch.load(
        ENCODED_DATA_PATH,
        map_location="cpu",
        weights_only=False,
    )

    print(
        "Loaded encoded data from:",
        ENCODED_DATA_PATH,
    )


num_sessions = len(train["samples"])
num_bars = train["samples"][0][0].shape[0]
num_assets = len(train["asset_cols"])
num_origins = num_bars - CONTEXT_LENGTH + 1

assert tuple(
    encoded_data["context_s1"].shape
) == (
    num_sessions,
    num_origins,
    CONTEXT_LENGTH,
    num_assets,
)

assert tuple(
    encoded_data["context_s2"].shape
) == (
    num_sessions,
    num_origins,
    CONTEXT_LENGTH,
    num_assets,
)

assert tuple(
    encoded_data["s1"].shape
) == (
    num_sessions,
    num_bars,
    num_assets,
)

assert tuple(
    encoded_data["s2"].shape
) == (
    num_sessions,
    num_bars,
    num_assets,
)

print(
    "Context-token shape:",
    tuple(encoded_data["context_s1"].shape),
)

print(
    "Final-token shape:",
    tuple(encoded_data["s1"].shape),
)

In [ ]:
if RUN_DECODER:
    decoded_data = decode_causal_split(
        kronos_tokenizer,
        encoded_data,
        window_batch_size=WINDOW_BATCH_SIZE,
        series_batch_size=SERIES_BATCH_SIZE,
        show_progress=True,
    )

    save_torch_atomic(
        decoded_data,
        DECODED_DATA_PATH,
    )

    print(
        "Saved decoded data to:",
        DECODED_DATA_PATH,
    )

else:
    if not DECODED_DATA_PATH.exists():
        raise FileNotFoundError(
            "RUN_DECODER is False but the decoded "
            f"cache does not exist: {DECODED_DATA_PATH}"
        )

    decoded_data = torch.load(
        DECODED_DATA_PATH,
        map_location="cpu",
        weights_only=False,
    )

    print(
        "Loaded decoded data from:",
        DECODED_DATA_PATH,
    )


assert tuple(
    decoded_data["decoded"].shape
) == (
    num_sessions,
    num_bars,
    num_assets,
    5,
)

assert tuple(
    decoded_data["valid_mask"].shape
) == (
    num_sessions,
    num_bars,
)

assert (
    decoded_data["valid_mask"][:, :59]
    .sum()
    .item()
    == 0
)

assert decoded_data[
    "valid_mask"
][:, 59:].all()

print(
    "Decoded shape:",
    tuple(decoded_data["decoded"].shape),
)

print(
    "Valid reconstructed bars:",
    int(
        decoded_data["valid_mask"]
        .sum()
        .item()
    ),
)

# Plot actual vs decoded

In [ ]:
figure, axis, selection = (
    plot_tokenizer_reconstruction(
        train,
        decoded_data,
        channel="close",
    )
)

selection